# Build a dataset class in a notebook

This notebook shows a notebook-first workflow for astronomers:

1. Start with required dataset methods for fixed-shape fields.
2. Verify structure with `h.prepare()` and a sample lookup.
3. Add a variable-length light-curve field plus custom `collate` padding/masking.
4. Move the class to an external package when it works.

## 1) Required dataset methods for fixed-shape fields

In [ ]:
import numpy as np
from hyrax.datasets import HyraxDataset


class NotebookSurveyDataset(HyraxDataset):
    def __init__(self, config, data_location=None, n_objects=64):
        rng = np.random.default_rng(7)
        self.images = rng.normal(size=(n_objects, 3, 32, 32)).astype(np.float32)
        self.labels = rng.integers(0, 5, size=n_objects, dtype=np.int64)
        super().__init__(config)

    def __len__(self):
        return len(self.images)

    def get_image(self, idx):
        return self.images[idx]

    def get_label(self, idx):
        return self.labels[idx]

    def get_object_id(self, idx):
        return f"obj-{idx:05d}"

## 2) Configure Hyrax and inspect one prepared sample

In [ ]:
from hyrax import Hyrax

h = Hyrax()
h.set_config("data_request", {
    "science": {
        "dataset_class": "NotebookSurveyDataset",
        "fields": ["image", "label", "object_id"],
        "primary_id_field": "object_id",
    }
})

prepared = h.prepare()
first = prepared[0]
first.keys(), first['science'].keys()

## 3) Add variable-length light curves + padding/mask collate

Now we add `get_light_curve` and custom `collate` so each batch has a consistent shape.

In [ ]:
class NotebookSurveyDatasetWithLightCurves(HyraxDataset):
    def __init__(self, config, data_location=None, n_objects=64, min_len=20, max_len=100):
        rng = np.random.default_rng(11)
        self.images = rng.normal(size=(n_objects, 3, 32, 32)).astype(np.float32)
        self.labels = rng.integers(0, 5, size=n_objects, dtype=np.int64)

        lengths = rng.integers(min_len, max_len + 1, size=n_objects)
        self.light_curves = [
            rng.normal(size=int(n)).astype(np.float32)
            for n in lengths
        ]

        super().__init__(config)

    def __len__(self):
        return len(self.images)

    def get_image(self, idx):
        return self.images[idx]

    def get_label(self, idx):
        return self.labels[idx]

    def get_light_curve(self, idx):
        return self.light_curves[idx]

    def get_object_id(self, idx):
        return f"obj-{idx:05d}"

    def collate(self, samples):
        images = np.stack([s["data"]["image"] for s in samples], axis=0).astype(np.float32)
        labels = np.array([s["data"]["label"] for s in samples], dtype=np.int64)

        curves = [s["data"]["light_curve"] for s in samples]
        max_len = max(len(c) for c in curves)

        light_curve = np.zeros((len(curves), max_len), dtype=np.float32)
        light_curve_mask = np.zeros((len(curves), max_len), dtype=np.float32)

        for i, c in enumerate(curves):
            n = len(c)
            light_curve[i, :n] = c
            light_curve_mask[i, :n] = 1.0

        object_ids = np.array([s["data"]["object_id"] for s in samples], dtype=str)

        return {
            "data": {
                "image": images,
                "label": labels,
                "light_curve": light_curve,
                "light_curve_mask": light_curve_mask,
                "object_id": object_ids,
            }
        }

In [ ]:
h2 = Hyrax()
h2.set_config("data_request", {
    "science": {
        "dataset_class": "NotebookSurveyDatasetWithLightCurves",
        "fields": ["image", "label", "light_curve", "object_id"],
        "primary_id_field": "object_id",
    }
})

prepared2 = h2.prepare()
raw_samples = [prepared2[i] for i in range(4)]
padded = prepared2.collate(raw_samples)

padded["science"]["light_curve"].shape, padded["science"]["light_curve_mask"].shape

## 4) Next step: move class into an external package

When this notebook version works, copy the class into your package and switch
`dataset_class` to its fully-qualified import path.